In [ ]:
# =============================================================================
# Toxic Language Detection — Hybrid Pipeline Notebook
# =============================================================================
# Must be placed inside: hybrid_pipeline-main/
# =============================================================================
import sys, os

# notebook.ipynb lives inside hybrid_pipeline-main/
# so we just add the current working directory directly
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), "experiments"))

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from ipywidgets import interact, Text

# Pipeline imports
from test_cases import test_cases
from hybrid import HybridPipeline
from boyer_moore import boyer_moore_detect
from aho_corasick import (
    aho_corasick_scan_prebuilt,
    build_trie,
    build_failure_links,
)


# =============================================================================
# Load dictionary from dataset
# =============================================================================
DATA_PATH = "data/toxic_word_dataset_final.xlsx"
df = pd.read_excel(DATA_PATH)
dictionary = (
    df.iloc[:, 0]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .tolist()
)

# =============================================================================
# Initialize Hybrid Pipeline
# =============================================================================
pipeline = HybridPipeline(dictionary)

# =============================================================================
# Run test cases through the pipeline
# =============================================================================
results = []

for tc in test_cases:
    res = pipeline.process(tc["message"])
    results.append({
        "Message": tc["message"],
        "Expected": tc["label"],
        "Detected": "toxic" if res["flagged"] else "clean",
        "Stage": res["stage_triggered"],
        "BM_Score": res.get("bm_score", 0),
        "Matches": ", ".join([m["pattern"] for m in res["matches"]]) if res["matches"] else "-",
    })

df_results = pd.DataFrame(results)
display(Markdown("### ✅ Detection Results (first 10)"))
display(df_results.head(10))

# =============================================================================
# Accuracy Summary
# =============================================================================
correct = (df_results["Detected"] == df_results["Expected"]).sum()
total = len(df_results)
accuracy = correct / total * 100

print(f"Pipeline detected {correct}/{total} messages correctly ({accuracy:.2f}%)")

# Add per-category accuracy barplot using seaborn
df_results["Correct"] = df_results["Detected"] == df_results["Expected"]
cat_acc = (
    pd.DataFrame(test_cases)
    .assign(Correct=df_results["Correct"])
    .groupby("type")["Correct"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 4))
sns.barplot(x=cat_acc.index, y=cat_acc.values, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Accuracy per Test Category")
plt.ylabel("Accuracy (0–1)")
plt.ylim(0, 1)
plt.show()

# =============================================================================
# Benchmark Results (Execution Time + Memory)
# =============================================================================
CSV_PATH = "experiments/results/tables/benchmark_results.csv"

if os.path.exists(CSV_PATH):
    bench = pd.read_csv(CSV_PATH)
    display(Markdown("### ⚙️ Benchmark Results Preview"))
    display(bench.head())

    # Compute averages
    avg = (
        bench.groupby("Approach")[["Time_us", "Memory_KB"]]
        .mean()
        .reset_index()
    )

    # --- Time Graph ---
    fig, ax1 = plt.subplots(figsize=(6, 4))
    avg.plot(
        kind="bar",
        x="Approach",
        y="Time_us",
        color=["#FF7777", "#77BBFF", "#77FF77"],
        ax=ax1,
    )
    plt.title("Average Execution Time per Algorithm (µs)")
    plt.ylabel("Time (µs)")
    plt.show()

    # --- Memory Graph ---
    fig, ax2 = plt.subplots(figsize=(6, 4))
    avg.plot(
        kind="bar",
        x="Approach",
        y="Memory_KB",
        color=["#FF9999", "#66B2FF", "#99FF99"],
        ax=ax2,
    )
    plt.title("Average Peak Memory per Algorithm")
    plt.ylabel("Memory (KB)")
    plt.show()
else:
    display(Markdown("⚠️ **No benchmark CSV found.** Run `experiments/benchmark.py` first."))

# =============================================================================
# Interactive Message Tester
# =============================================================================
display(Markdown("### 💬 Try your own message"))

def test_message(msg):
    """Interactive test input for toxic message detection."""
    out = pipeline.process(msg)
    label = "🔥 **Toxic**" if out["flagged"] else "✅ **Clean**"
    display(Markdown(label))
    if out["flagged"]:
        detected = [m["pattern"] for m in out["matches"]]
        print("Detected keywords:", detected)

interact(test_message, msg=Text(value="type your message here"))

display(Markdown("Notebook completed successfully. ✅"))


ModuleNotFoundError: No module named 'experiments.test_cases'